## Course map

| Notebook | Main focus |
|---|---|
| Beginner | Installation, ONNX validation, JSON basics, first compile, Model Zoo |
| Intermediate | Calibration quality, hardware PPU, YOLO26 TopK optimization |
| Advanced | Q-PRO, diagnosis, QXNN resume, QAT, Python API, dvanced compiler controls |

Complete the notebooks in order unless you already understand DX-COM configuration and calibration.


# DX-COM Tutorial 1: Beginner

This notebook introduces the complete ONNX-to-DXNN workflow with small, reproducible examples.

## Learning objectives

By the end of this tutorial, you will be able to:

1. understand the DX-COM overall process,
2. verify the DX-COM installation,
3. export and validate a MobileNetV2 ONNX model,
4. create a calibration configuration,
5. compile and inspect a DXNN model, and
6. download an ONNX model and its JSON configuration from the DEEPX Model Zoo and compile it.

This notebook does not modify the SDK source tree. All generated files are stored under `<dx-tutorials>/notebooks/T05-DX-Compiler`.


## 1. Compilation workflow

The overall compilation process consists of the following three steps:

1. **Obtain a pretrained model** — train a model or obtain a compatible pretrained model from a framework such as PyTorch.
2. **Convert the model to ONNX** — export the framework model to ONNX and verify its input names, input shapes, operators, and outputs.
3. **Compile ONNX to DXNN** — use DX-COM with a JSON configuration and representative calibration data to generate a model for the DEEPX NPU.

<img src="assets/dx-com-workflow.jpg" style="max-width: 800px; width: 100%;" alt="DX-COM compilation workflow from a pretrained model to ONNX and DXNN">

This beginner tutorial follows the same workflow with MobileNetV2: PyTorch → ONNX → DXNN.

For more details, refer to the DX-Compiler user guide 👉 [Download](https://developer.deepx.ai/download/?id=581)

> Note: To download the User Guide, you must log in to https://developer.deepx.ai/ first.

### Files used and created during compilation

Keep these artifacts separate in your mind. Each one has a different role in the workflow.

| Artifact | Role | Used or created by |
|---|---|---|
| Framework model, such as `.pt` | Trained weights and network definition from the original framework | Used when exporting ONNX |
| `.onnx` | Framework-independent model graph that DX-COM reads | Input to DX-COM |
| Compiler `.json` | Input shape, calibration, preprocessing, quantization, and optional compiler settings | Input to DX-COM |
| Calibration dataset | Representative samples used to estimate quantization ranges | Read by DX-COM during calibration |
| `.dxnn` | Compiled model executed by the DEEPX runtime | Main DX-COM output |
| `compiler.log` | Detailed compilation messages, warnings, and errors | Created with `--gen_log` |
| `*_summary.html` | Visual compilation report containing graph and compiler information | Created with `--export_html` |

The `.onnx`, compiler `.json`, and calibration dataset are compilation inputs. The `.dxnn`, log, and HTML report are outputs.

<img src="assets/dx-compile-progress.png" style="max-width: 1000px; width: 100%;" alt="DX-COM compilation with required files">

## 2. Requirements and workspace

DX-COM supports **x86-64 Linux**. Use a static **ONNX input** shape with **batch size 1**. A practical host has at least **16 GB of RAM** and 8 GB of free storage.

The following cell reads the SDK location from `config.json` through `tutorial_paths.py`. It then derives the compiler and virtual-environment paths automatically.


In [ ]:
from pathlib import Path
import json
import os
import shlex
import sys

root_path = os.environ.get("ROOT_PATH")
if not root_path:
    raise EnvironmentError("ROOT_PATH is not set. Start JupyterLab with ./run-jupyter-lab.sh")
%run "$root_path/tutorial_paths.py"
print_tutorial_paths()

DX_COM_DIR = DX_COMPILER_DIR / "dx_com"
DX_COMPILER_VENV = DX_COMPILER_DIR / "venv-dx-compiler-local"
DXCOM_PATH = DX_COMPILER_VENV / "bin" / "dxcom"
DXCOM_PYTHON = DX_COMPILER_VENV / "bin" / "python"

for required_path in (DX_COM_DIR, DXCOM_PATH, DXCOM_PYTHON):
    if not required_path.exists():
        raise FileNotFoundError(f"Required DX-COM path does not exist: {required_path}")

WORK_DIR = TUTORIAL_ROOT / "notebooks/T05-DX-Compiler"
MODEL_DIR = WORK_DIR / "models"
CONFIG_DIR = WORK_DIR / "configs"
OUTPUT_DIR = WORK_DIR / "outputs"

for path in (WORK_DIR, MODEL_DIR, CONFIG_DIR, OUTPUT_DIR):
    path.mkdir(parents=True, exist_ok=True)

# Reuse the SDK sample images without copying them into this tutorial workspace.
CALIBRATION_SOURCE = DX_COM_DIR / "calibration_dataset"
CALIBRATION_DIR = WORK_DIR / "calibration_dataset"
if not CALIBRATION_SOURCE.is_dir():
    raise FileNotFoundError(f"Calibration dataset was not found: {CALIBRATION_SOURCE}")
if not CALIBRATION_DIR.exists():
    CALIBRATION_DIR.symlink_to(CALIBRATION_SOURCE, target_is_directory=True)

os.chdir(WORK_DIR)
print(f"DX-COM       : {DXCOM_PATH}")
print(f"Workspace    : {WORK_DIR}")
print(f"Calibration  : {CALIBRATION_DIR} -> {CALIBRATION_SOURCE}")


### 2.1 DX-COM installation options

DX-COM is a Python package, and installing it in a **dedicated Python virtual environment is strongly recommended**. A virtual environment isolates DX-COM and its dependencies from the system Python and the Jupyter kernel, which reduces version conflicts and makes the compiler environment easier to reproduce or remove.

The easiest standalone installation uses the package published on [PyPI](https://pypi.org/project/dx-com/). Run the following commands in a separate terminal:

```bash
# 1. Create a dedicated environment.
python3 -m venv ~/venv-dx-com

# 2. Activate it in the current terminal.
source ~/venv-dx-com/bin/activate

# 3. Install DX-COM from PyPI.
python -m pip install --upgrade pip
pip install dx-com

# 4. Confirm that the virtual environment owns the commands.
which python
which dxcom
dxcom --version
```

The two `which` commands should print paths under `~/venv-dx-com/`. Run `source ~/venv-dx-com/bin/activate` again whenever you open a new terminal, and run `deactivate` when you finish.

This notebook uses the dedicated environment created by `dx-all-suite/dx-compiler/install.sh` at `venv-dx-compiler-local`, so its compiler does not depend on the Jupyter kernel's Python environment. The PyPI procedure above is the recommended alternative when installing DX-COM separately.

> Installing `dx-com` provides the compiler package and the `dxcom` command. The rest of the DEEPX SDK and a supported Linux x86-64 host are still required for the complete compile-and-deploy workflow.


The next cell is equivalent to these terminal commands:

```bash
source ~/dx-all-suite/dx-compiler/venv-dx-compiler-local/bin/activate
dxcom --version
dxcom -h
```

The actual SDK path may differ; the code cell uses the path loaded from `config.json`.


In [ ]:
# Check the version of DX-Compiler
!source "{DX_COMPILER_VENV}/bin/activate" && dxcom --version


In [ ]:
# Show dxcom usages with --help/-h option
!source "{DX_COMPILER_VENV}/bin/activate" && dxcom -h


## 3. Export a small ONNX model

This project uses a `uv`-managed Jupyter environment, which may not contain the `pip` module.

`uv pip install --python "{sys.executable}"` installs packages into the current Jupyter kernel explicitly. 

These packages are used to export and inspect ONNX. DX-COM itself continues to run from `DX_COMPILER_VENV`.


In [ ]:
import sys

!uv pip install --python "{sys.executable}" --quiet "torch>=2.0" torchvision onnx


In [ ]:
import torch
import torchvision
import onnx

torch.manual_seed(0)
weights = torchvision.models.MobileNet_V2_Weights.DEFAULT
model = torchvision.models.mobilenet_v2(weights=weights).eval()
dummy_input = torch.randn(1, 3, 224, 224)
mobilenet_onnx = MODEL_DIR / "mobilenet_v2.onnx"

torch.onnx.export(
    model,
    dummy_input,
    mobilenet_onnx,
    input_names=["input"],
    output_names=["output"],
    opset_version=18,
    do_constant_folding=True,
)
print(mobilenet_onnx)


### 3.1 Validate the ONNX contract

The input name in the JSON file must exactly match the ONNX graph input name. Shape inference and `onnx.checker` catch many export errors before compilation.


In [ ]:
onnx_model = onnx.load(mobilenet_onnx)
onnx.checker.check_model(onnx_model)
print("Opset:", [(item.domain or "ai.onnx", item.version) for item in onnx_model.opset_import])

def tensor_shape(value_info):
    return [dim.dim_value or dim.dim_param for dim in value_info.type.tensor_type.shape.dim]

print("Inputs:")
for value in onnx_model.graph.input:
    print(f"  {value.name}: {tensor_shape(value)}")
print("Outputs:")
for value in onnx_model.graph.output:
    print(f"  {value.name}: {tensor_shape(value)}")


You can also open the ONNX file in [Netron](https://netron.app/) to inspect tensor names, shapes, and operators. DX-TRON is deprecated in DX-COM 2.4.0; use Netron before compilation and the DX-COM HTML summary after compilation.


## 4. Create the calibration configuration

Calibration images are very important to keep high accuracy during the quantization process.

<img src="assets/calibration.jpeg" style="max-width: 1000px;" alt="Representative calibration images used during quantization">

The configuration describes both the model input contract and how source images become input tensors.

| Field | Purpose |
|---|---|
| `inputs` | Exact ONNX input name and static shape |
| `calibration_method` | Observer used to estimate quantization ranges |
| `calibration_num` | Number of representative samples to use |
| `default_loader.dataset_path` | Directory containing calibration inputs |
| `preprocessings` | Ordered image-to-tensor transformations |

Preprocessing order matters. The values must match the preprocessing used when the original model was trained and evaluated.


In [ ]:
mobilenet_config = {
    "inputs": {"input": [1, 3, 224, 224]},
    "calibration_method": "ema",
    "calibration_num": 100,
    "default_loader": {
        "dataset_path": "./calibration_dataset",
        "file_extensions": ["jpeg", "jpg", "png", "JPEG"],
        "preprocessings": [
            {"resize": {"mode": "torchvision", "size": 256, "interpolation": "BILINEAR"}},
            {"centercrop": {"width": 224, "height": 224}},
            {"convertColor": {"form": "BGR2RGB"}},
            {"div": {"x": 255.0}},
            {"normalize": {
                "mean": [0.485, 0.456, 0.406],
                "std": [0.229, 0.224, 0.225]
            }},
            {"transpose": {"axis": [2, 0, 1]}},
            {"expandDim": {"axis": 0}}
        ]
    }
}

mobilenet_config_path = CONFIG_DIR / "mobilenet_v2.json"
mobilenet_config_path.write_text(json.dumps(mobilenet_config, indent=2) + "\n")
print(mobilenet_config_path.read_text())


### 4.1 Preflight checklist before compilation

Run these checks before starting a potentially long compilation:

- the ONNX file passes `onnx.checker`;
- every input dimension is static;
- ONNX input names and shapes exactly match the compiler JSON;
- the calibration directory exists and contains supported files;
- calibration preprocessing matches the model's training and evaluation preprocessing; and
- the selected DX-COM executable exists.

> **Critical requirement — batch size must be `1`.** Both the ONNX input and the JSON `inputs` entry must start with `1`, for example `[1, 3, 224, 224]`.
>
> A DXNN model targets the DEEPX NPU's single-sample execution contract, so a different or dynamic batch dimension does not match the compilation and runtime contract. To improve throughput, submit multiple inference requests or use an asynchronous pipeline; do not increase the model's batch dimension.

The next cell turns the checklist into executable checks and stops before DX-COM if a requirement is not satisfied.

In [ ]:
onnx.checker.check_model(onnx_model)

onnx_inputs = {
    value.name: tensor_shape(value)
    for value in onnx_model.graph.input
}
json_inputs = mobilenet_config.get("inputs", {})

if set(onnx_inputs) != set(json_inputs):
    raise ValueError(
        f"Input-name mismatch: ONNX={sorted(onnx_inputs)}, "
        f"JSON={sorted(json_inputs)}"
    )

for input_name, onnx_shape in onnx_inputs.items():
    json_shape = json_inputs[input_name]
    if any(not isinstance(dimension, int) or dimension <= 0 for dimension in onnx_shape):
        raise ValueError(f"ONNX input must be static: {input_name}={onnx_shape}")
    if onnx_shape[0] != 1 or json_shape[0] != 1:
        raise ValueError(
            f"Batch size must be 1: ONNX={onnx_shape[0]}, JSON={json_shape[0]}"
        )
    if onnx_shape != json_shape:
        raise ValueError(
            f"Input-shape mismatch for {input_name}: "
            f"ONNX={onnx_shape}, JSON={json_shape}"
        )

dataset_value = Path(mobilenet_config["default_loader"]["dataset_path"]).expanduser()
calibration_path = (WORK_DIR / dataset_value).resolve() if not dataset_value.is_absolute() else dataset_value.resolve()
if not calibration_path.is_dir():
    raise FileNotFoundError(f"Calibration directory was not found: {calibration_path}")

extensions = {
    "." + extension.lower().lstrip(".")
    for extension in mobilenet_config["default_loader"]["file_extensions"]
}
calibration_files = [
    path for path in calibration_path.rglob("*")
    if path.is_file() and path.suffix.lower() in extensions
]
if not calibration_files:
    raise FileNotFoundError(f"No supported calibration files were found in {calibration_path}")
if not DXCOM_PATH.is_file():
    raise FileNotFoundError(f"DX-COM executable was not found: {DXCOM_PATH}")

print("PASS: ONNX model is valid")
print(f"PASS: input contract matches: {onnx_inputs}")
print("PASS: batch size = 1 (required for DXNN)")
print(f"PASS: calibration files = {len(calibration_files)}")
print(f"PASS: DX-COM executable = {DXCOM_PATH}")


## 5. Compile to DXNN

### 5.1. Compile

`--gen_log` preserves compiler logs and `--export_html` creates a model summary. The command keeps errors visible and uses a dedicated output directory.

The next cell is equivalent to these terminal commands:

```bash
source ~/dx-all-suite/dx-compiler/venv-dx-compiler-local/bin/activate
dxcom -m models/mobilenet_v2.onnx \
      -c configs/mobilenet_v2.json \
      -o outputs/mobilenet_v2_q_lite \
      --gen_log \
      --export_html
```

In [ ]:
mobilenet_output = OUTPUT_DIR / "mobilenet_v2_q_lite"

!source "{DX_COMPILER_VENV}/bin/activate" && \
  dxcom -m "{mobilenet_onnx}" \
        -c "{mobilenet_config_path}" \
        -o "{mobilenet_output}" \
        --gen_log \
        --export_html

In [ ]:
generated = sorted(path.relative_to(WORK_DIR) for path in mobilenet_output.rglob("*"))
print("\n".join(map(str, generated)))
dxnn_candidates = list(mobilenet_output.glob("*.dxnn"))
if not dxnn_candidates:
    raise FileNotFoundError("Compilation did not produce a DXNN file. Review the compiler output above.")
mobilenet_dxnn = dxnn_candidates[0]
print(f"\nDXNN: {mobilenet_dxnn}")


### 5.2. Inspect and benchmark

`dxparse -v` reports the compiled model structure and tensor metadata. `dxrun --use-ort -t 5` then performs a five-second synthetic-input benchmark.


In [ ]:
!dxparse -m "{mobilenet_dxnn}" -v


In [ ]:
!dxrun -m "{mobilenet_dxnn}" --use-ort -t 5


If the compiler reports `[INFO] Added nodes`, inspect which preprocessing operations were inserted into the graph. Do not apply the same normalization, color conversion, or transpose again in the runtime application.


### 5.3. Performance benchmark and accuracy evaluation are different

| | `dxrun` synthetic benchmark | Dataset accuracy evaluation |
|---|---|---|
| Input | Generated dummy input | Real, labeled validation dataset |
| Main result | Runtime throughput and latency | Task metric such as Top-1, Top-5, mAP, or mIoU |
| What it checks | The compiled model can execute and its runtime performance | Preprocessing, inference, post-processing, and prediction quality |
| What it does **not** prove | Model accuracy | Deployment latency under the final application workload |

> **A successful compile and a fast `dxrun` result do not prove model accuracy.**

For accuracy measurement, refer to [DEEPX-AI/dx-modelzoo](https://github.com/DEEPX-AI/dx-modelzoo) and its [Model Evaluation guide](https://github.com/DEEPX-AI/dx-modelzoo/blob/main/docs/source/guides/evaluation.md). DX-ModelZoo connects the dataset, preprocessing, runtime profile, post-processing, and evaluator through a model configuration and reports task-specific metrics. Use the same labeled validation set when comparing ONNX and DXNN results.

## 6. Compile a model from the DEEPX Model Zoo

The [DEEPX Model Zoo](https://developer.deepx.ai/modelzoo/) provides searchable model metadata and downloadable artifacts, including ONNX models, DXNN models, and compiler JSON files for supported quantization variants.

This exercise uses **Resnet50** because its ONNX file is small. The workflow is the same for a larger model:

1. choose a model and quantization variant,
2. download its ONNX and matching JSON,
3. inspect the ONNX input contract,
4. adapt environment-specific JSON values, and
5. compile into a new output directory.


### 6.1. Download a Resnet50 ONNX file and a dxcom configuration file (json)

In [ ]:
MODELZOO_ONNX_URL = "https://sdk.deepx.ai/modelzoo/onnx/resnet50_224x224.onnx"
MODELZOO_JSON_URL = "https://sdk.deepx.ai/modelzoo/q-lite-json/2_4_0/resnet50_224x224.json"
MODELZOO_ONNX = MODEL_DIR / "resnet50_224x224.onnx"
MODELZOO_JSON = CONFIG_DIR / "resnet50_224x224.json"

!wget --continue --output-document="{MODELZOO_ONNX}" "{MODELZOO_ONNX_URL}"
!wget --continue --output-document="{MODELZOO_JSON}" "{MODELZOO_JSON_URL}"


In [ ]:
modelzoo_model = onnx.load(MODELZOO_ONNX)
onnx.checker.check_model(modelzoo_model)
print("ONNX inputs:")
for value in modelzoo_model.graph.input:
    print(f"  {value.name}: {tensor_shape(value)}")

downloaded_config = json.loads(MODELZOO_JSON.read_text())
print("Downloaded dataset path:", downloaded_config["default_loader"]["dataset_path"])
print("Configured input:", downloaded_config["inputs"])


### 6.2. Update the calibration path of dxcom configuration file (json)

Model Zoo JSON files contain the calibration setup used to produce the published model. Their dataset path belongs to the build environment and will usually not exist on your computer. Keep the model-specific preprocessing, but replace the dataset path with your local representative dataset.

The SDK sample images are used here only to make the compiler workflow reproducible. For an accuracy decision, use samples from the real deployment domain.


In [ ]:
local_modelzoo_config = downloaded_config.copy()
local_modelzoo_config["default_loader"] = downloaded_config["default_loader"].copy()
local_modelzoo_config["default_loader"]["dataset_path"] = "./calibration_dataset"

MODELZOO_LOCAL_JSON = CONFIG_DIR / "resnet50_224x224.json"
MODELZOO_LOCAL_JSON.write_text(json.dumps(local_modelzoo_config, indent=2) + "\n")
print(MODELZOO_LOCAL_JSON.read_text())


### 6.3. Compile

The next cell is equivalent to these terminal commands:

```bash
source ~/dx-all-suite/dx-compiler/venv-dx-compiler-local/bin/activate
dxcom -m models/resnet50_224x224.onnx \
      -c configs/resnet50_224x224.json \
      -o outputs/resnet50_224x224_q_lite \
      --gen_log \
      --export_html
```

In [ ]:
modelzoo_output = OUTPUT_DIR / "resnet50_224x224_q_lite"

!source "{DX_COMPILER_VENV}/bin/activate" && \
  dxcom -m "{MODELZOO_ONNX}" \
        -c "{MODELZOO_LOCAL_JSON}" \
        -o "{modelzoo_output}" \
        --gen_log \
        --export_html


### 6.4. Inspect and benchmark

In [ ]:
modelzoo_dxnn_files = list(modelzoo_output.glob("*.dxnn"))
if not modelzoo_dxnn_files:
    raise FileNotFoundError("No DXNN file was generated.")
modelzoo_dxnn = modelzoo_dxnn_files[0]
!dxparse -m "{modelzoo_dxnn}" -v
!dxrun -m "{modelzoo_dxnn}" --use-ort -t 5


### 6.5 Open the latest HTML compilation report

DX-COM creates an HTML model summary when `--export_html` is enabled. The next cell finds the most recently generated report under this tutorial's output directory and displays a button that opens it in a new browser tab.


In [ ]:
from html import escape
from urllib.parse import quote
from IPython.display import HTML, display

html_reports = [path for path in OUTPUT_DIR.rglob("*.html") if path.is_file()]
if not html_reports:
    raise FileNotFoundError(
        "No DX-COM HTML report was found. "
        "Compile a model with --export_html first."
    )

latest_report = max(html_reports, key=lambda path: path.stat().st_mtime)
relative_report = latest_report.resolve().relative_to(TUTORIAL_ROOT.resolve())
report_url = "/files/" + quote(relative_report.as_posix(), safe="/")

display(HTML(
    f'<a href="{escape(report_url, quote=True)}" target="_blank" '
    'rel="noopener noreferrer" '
    'style="display:inline-block;padding:10px 16px;background:#2563eb;'
    'color:white;text-decoration:none;border-radius:6px;font-weight:600;">'
    f'Open DX-COM report: {escape(latest_report.name)}'
    '</a>'
))
print(f"Report: {latest_report}")


## 7. Troubleshooting checklist

- **`dxcom: command not found`**: activate `DX_COMPILER_VENV`, or call `DXCOM_PATH` directly.
- **Input key error**: compare `inputs` in JSON with `model.graph.input` exactly.
- **Dynamic shape or batch error**: export a static shape with batch size 1.
- **No calibration files**: check the path, extensions, and file permissions.
- **Unexpected accuracy loss**: verify channel order, scaling, normalization, resize policy, and dataset representativeness.
- **Runtime preprocessing mismatch**: check whether DX-COM inserted preprocessing nodes.

Keep each experiment in its own output directory. This makes compiler reports and binaries traceable.


## 8. Summary

### 8.1. The workflow you completed

**PyTorch model**  
→ **ONNX export and validation**  
→ **JSON configuration and calibration data**  
→ **DX-COM compilation**  
→ **DXNN inspection and benchmark**

<img src="assets/dx-compile-progress.png"
     style="max-width: 1000px; width: 100%;"
     alt="DX-COM compilation workflow">

### 8.2. Inputs and outputs

| Stage | Main artifact | Verification |
|---|---|---|
| Model preparation | `mobilenet_v2.onnx` | `onnx.checker` |
| Compiler configuration | `mobilenet_v2.json` | Preflight checklist |
| Compilation | `mobilenet_v2.dxnn` | Output file check |
| Structure inspection | DXNN metadata | `dxparse -v` |
| Performance check | Dummy-input benchmark | `dxrun` |
| Accuracy evaluation | Labeled validation dataset | DX-ModelZoo |

### 8.3. Completion checklist

- [x] Exported a PyTorch model to ONNX
- [x] Verified the ONNX input name and static shape
- [x] Confirmed that the batch size is `1`
- [x] Created a calibration configuration
- [x] Compiled ONNX into DXNN
- [x] Inspected the compiled model with `dxparse`
- [x] Measured runtime performance with `dxrun`
- [ ] Evaluate model accuracy with a labeled dataset

> **Remember:** successful compilation and a fast `dxrun` result do not
> prove model accuracy. Accuracy must be measured separately with
> representative labeled data.

### Next step

Continue with the **Intermediate tutorial** to learn:

- calibration dataset design;
- Type 0 and Type 1 hardware PPU;
- PPU and non-PPU performance comparison; and
- YOLO26 TopK-based optimization.